# Cathey — Development & Debug Notebook
Interactive testing of the full pipeline. Run cells top-to-bottom for complete setup, or jump to any section.

**Sections**
1. Setup & Imports
2. LLM — Classification · Clarification · QA
3. Rule-Based Fast Path
4. Schema Validation
5. Memory System
6. Full Agent Pipeline (text input)
7. Quick STT Test
8. TTS Test
9. Microphone Test — Mac
10. Microphone Test — Windows


## 1. Setup & Imports

In [1]:
import sys, os, logging, time
logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
logging.getLogger('faster_whisper').setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer
from llm_parser import LLMParser
from memory import MemoryManager
from agent import CatheyAgent
from rule_based import try_rule_based
from schema import validate_command, execute_command
from config import EMBED_MODEL_NAME
print('Imports OK')


Imports OK


In [2]:
print('Loading LLM ...')
llm = LLMParser()

print('Loading embedding model ...')
embed = SentenceTransformer(EMBED_MODEL_NAME)
embed.encode('warmup', convert_to_numpy=True)

memory = MemoryManager(embed_fn=lambda t: embed.encode(t, convert_to_numpy=True).tolist())

try:
    from gpio_executor import GPIOExecutor
    gpio = GPIOExecutor()
    print('GPIO ready (Pi)')
except Exception as e:
    gpio = None
    print(f'GPIO unavailable ({e}) — using stub')

def speak(text): print(f'[TTS] {text}')

agent = CatheyAgent(llm=llm, memory=memory, speak=speak, gpio=gpio)
print('\nAll components ready.')


Loading LLM ...
Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapter from cathey_lora_adapter/final_adapter ...
LoRA adapter loaded.
LLM ready.
Loading embedding model ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

GPIO unavailable (No module named 'lgpio') — using stub

All components ready.


## 2. LLM — Classification · Clarification · QA

Three cells covering all three LLM inference methods. Edit the test inputs as needed.

In [3]:
# Intent classification — one representative case per type + an edge case
def classify(text):
    result, raw, ms = llm.parse_unified(text, verbose=False)
    print(f'Input : {text}')
    print(f'Output: {result}')
    print(f'Raw   : {raw}')
    print(f'Time  : {ms:.0f} ms\n')
    return result

test_inputs = [
    'Cathey, turn on the light.',          # direct_command
    'Cathey, set the AC to 24 degrees.',   # direct_command (with value)
    'Cathey, I feel cold.',                # needs_clarification
    "Cathey, it's too bright.",            # needs_clarification
    'Cathey, how do I make tea?',          # general_qa
    'Cathey, hello!',                      # general_qa (greeting)
    'Turn on the light.',                  # invalid (no wake word)
]
for t in test_inputs:
    classify(t)


Input : Cathey, turn on the light.
Output: {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': None, 'reply': 'Turning on the light.'}
Raw   : {"type":"direct_command","device":"light","action":"turn_on","value":null,"reply":"Turning on the light."}
Time  : 5294 ms

Input : Cathey, set the AC to 24 degrees.
Output: {'type': 'direct_command', 'device': 'ac', 'action': 'set_temperature', 'value': 24, 'reply': 'Setting AC to 24 degrees.'}
Raw   : {"type":"direct_command","device":"ac","action":"set_temperature","value":24,"reply":"Setting AC to 24 degrees."}
Time  : 2865 ms

Input : Cathey, I feel cold.
Output: {'type': 'needs_clarification', 'question': 'Would you like me to close the window or raise the AC temperature?', 'options': ['close_window', 'raise_ac_temperature'], 'reply': 'Would you like me to close the window or raise the AC temperature?'}
Raw   : {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperatur

In [ ]:
# Clarification follow-up resolution
def resolve(original, question, options, reply):
    result, raw, ms = llm.resolve_followup(
        original_text=original, question=question,
        options=options, user_reply=reply, verbose=False)
    print(f'Reply : {reply}')
    print(f'Output: {result}')
    print(f'Time  : {ms:.0f} ms\n')
    return result

resolve(
    original="Cathey, I feel cold.",
    question="Would you like me to close the window or raise the AC temperature?",
    options=["close_window", "raise_ac_temperature"],
    reply="close the window please"
)
resolve(
    original="Cathey, it's too bright.",
    question="Would you like me to lower the brightness or close the curtain?",
    options=["lower_brightness", "close_curtain"],
    reply="the second one"
)


In [ ]:
# General QA — basic answer + memory-context answer
def qa(question, context=''):
    answer, ms = llm.answer_qa(question, context=context, verbose=False)
    print(f'Q: {question}')
    print(f'A: {answer}')
    print(f'Time: {ms:.0f} ms\n')
    return answer

qa('Cathey, how do I make tea?')

# With memory context (tests RAG-augmented answer)
ctx = memory.build_context("what's my name?")
if ctx:
    print('Memory context:\n', ctx[:200], '\n')
    qa("Cathey, what's my name?", context=ctx)
else:
    print('(memory empty — run some agent interactions first to populate)')


## 3. Rule-Based Fast Path

In [ ]:
rule_tests = [
    'Cathey, turn on the light.',
    'Cathey, open the curtain halfway.',
    'Cathey, set the AC to 24 degrees.',
    'Cathey, set brightness to 70.',
    'Cathey, make the light warmer.',
    'Cathey, party time!',
    'Cathey, I feel cold.',     # → LLM (no rule match)
    'Cathey, what time is it?', # → LLM
]

state = gpio.get_device_state() if gpio else {}
for t in rule_tests:
    result = try_rule_based(t, state)
    tag = '✓ RULE' if result else '→ LLM '
    print(f'{tag}  {t!r:50}  →  {result}')


## 4. Schema Validation

In [ ]:
test_commands = [
    {'device': 'light',   'action': 'turn_on',        'value': None},   # ✓
    {'device': 'light',   'action': 'set_brightness',  'value': 80},    # ✓
    {'device': 'curtain', 'action': 'set_position',    'value': 50},    # ✓
    {'device': 'ac',      'action': 'set_temperature', 'value': 24},    # ✓
    {'device': 'ac',      'action': 'set_temperature', 'value': None},  # ✗ needs int
    {'device': 'light',   'action': 'set_brightness',  'value': 150},   # ✗ out of range
    {'device': 'none',    'action': 'turn_on',         'value': None},  # ✗ unknown device
]

for cmd in test_commands:
    ok, reason = validate_command(cmd)
    print(f"{'✓' if ok else '✗'} {str(cmd):60} → {reason}")


## 5. Memory System

In [ ]:
import json

# Summary
print('── Memory summary ─────────────────────────────')
print(memory.summary())

# Semantic (user prefs)
print('\n── User preferences (semantic memory) ─────────')
print(memory.prefs)

# Procedural (skills)
print('\n── Learned skills (procedural memory) ──────────')
print(json.dumps(memory.skills, indent=2))

# Episodic retrieval
print('\n── Episodic retrieval (top 3 for "what is my name") ──')
if memory.episodes.count() > 0:
    eps = memory.retrieve_episodes("what's my name?", n=3)
    for ep in eps:
        print(f"  dist={ep['distance']:.3f}  {ep['text'][:50]}  →  {ep['meta']['cathey_reply'][:40]}")
else:
    print('  (episodic memory empty)')


## 6. Full Agent Pipeline (text input)

In [ ]:
def run(text):
    print(f'>>> {text}')
    result = agent.handle(text, verbose=True)
    print(f"Result: valid={result['valid']}  reason={result['reason']}  exec={result.get('execution')}\n")
    return result

# Direct commands (fast-path rule-based)
run('Cathey, turn on the light.')
run('Cathey, set the AC to 26 degrees.')
run('Cathey, open the curtain halfway.')


In [ ]:
# needs_clarification → procedural memory auto-resolve (if trained) or ask
run('Cathey, I feel cold.')

# general_qa
run('Cathey, how do I make tea?')

# invalid — no wake word (prefilter blocks it)
run('Turn on the light.')


## 7. Quick STT Test

Records 3 s and transcribes via Whisper. Works on Mac and Pi.
For a full mic test with agent integration, see sections 9 and 10.

In [ ]:
try:
    from audio import STTModel
    import sounddevice as sd
    import numpy as np
    from config import SAMPLE_RATE

    stt = STTModel()
    duration = 3
    print(f'Recording {duration}s — speak now...')
    audio = sd.rec(int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE,
                   channels=1, dtype='float32')
    sd.wait()
    text = stt.transcribe(audio)
    print('Transcription:', text)
except Exception as e:
    print(f'STT unavailable: {e}')


## 8. TTS Test

Piper TTS — runs on Pi. On Mac the stub `speak()` defined in setup prints instead.

In [ ]:
try:
    from audio import TTSEngine
    tts_engine = TTSEngine()
    tts_engine.speak('Hello, I am Cathey, your smart home assistant.')
except Exception as e:
    print(f'TTS unavailable: {e}')


## 9. Microphone Test — Mac

Records 4 s from the default microphone, runs STT, then passes the transcript to the agent.

**Requirements (Mac)**
```bash
pip install sounddevice numpy soundfile faster-whisper
# If sounddevice fails: brew install portaudio  then re-run pip install sounddevice
```

**Prerequisites**: run cells in Section 1 first.

In [ ]:
# ── Mac Microphone Test ──────────────────────────────────────────────────────
import platform
print(f'Platform: {platform.system()} {platform.mac_ver()[0] or "N/A"}')

try:
    import sounddevice as sd
    import numpy as np
    from config import SAMPLE_RATE
except ImportError as e:
    print(f'[ERROR] {e}\nInstall: pip install sounddevice numpy')
    print('On Mac:  brew install portaudio   (if above fails)')
    raise

# 1) List input devices
print('\n── Available microphone devices ─────────────────────────────')
for i, dev in enumerate(sd.query_devices()):
    if dev['max_input_channels'] > 0:
        default = ' ◄ default' if i == sd.default.device[0] else ''
        print(f"  [{i:2d}] {dev['name']:<45} ch={dev['max_input_channels']}  sr={int(dev['default_samplerate'])}{default}")
print()

# 2) Record  — change DEVICE to an index from the list above if needed
DURATION = 4
DEVICE   = None   # None = system default
print(f'Recording {DURATION}s — speak now!  (e.g. \'Cathey, turn on the light.\')')
audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE,
               channels=1, dtype='float32', device=DEVICE)
sd.wait()
print('Done.\n')

# 3) Energy check
energy = float(np.mean(np.abs(audio)))
print(f'Energy level : {energy:.6f}')
if energy < 0.003:
    print('⚠  Very quiet — check System Settings → Sound → Input.')
elif energy < 0.01:
    print('△  Low — move closer to the mic.')
else:
    print('✓  Good audio level.')

# 4) STT + Agent
try:
    from audio import STTModel
    stt = STTModel()
    text = stt.transcribe(audio)
    print(f'\nTranscription: {text!r}')
    if text:
        print('\n── Agent result ─────────────────────────────────────────────')
        result = agent.handle(text, verbose=True)
        print(f"valid={result['valid']}  reason={result['reason']}  exec={result.get('execution')}")
except Exception as e:
    print(f'\n[STT/Agent ERROR] {e}')


## 10. Microphone Test — Windows

Primary backend: `sounddevice` (PortAudio DLLs bundled on Windows — usually works out of the box).  
Auto-fallback: `pyaudio` if sounddevice is unavailable.

**Requirements (Windows)**
```bash
pip install sounddevice numpy soundfile faster-whisper   # try this first
# If sounddevice fails:
pip install pyaudio
# If pip install pyaudio fails:
pip install pipwin && pipwin install pyaudio
```

**Prerequisites**: run cells in Section 1 first.

In [ ]:
# ── Windows Microphone Test ───────────────────────────────────────────────────
import platform
print(f'Platform: {platform.system()} {platform.version()[:60]}')

SAMPLE_RATE = 16000
DURATION    = 4

# ── Try sounddevice first ─────────────────────────────────────────────────────
_sd_ok = False
try:
    import sounddevice as sd
    import numpy as np
    _sd_ok = True
    print('Audio backend : sounddevice ✓')
except ImportError:
    print('sounddevice not found — switching to pyaudio fallback ...')

if _sd_ok:
    print('\n── Available microphone devices ─────────────────────────────')
    for i, dev in enumerate(sd.query_devices()):
        if dev['max_input_channels'] > 0:
            default = ' ◄ default' if i == sd.default.device[0] else ''
            print(f"  [{i:2d}] {dev['name']:<45} ch={dev['max_input_channels']}{default}")
    print()
    DEVICE = None   # set to a specific index if the wrong mic is selected
    print(f'Recording {DURATION}s — speak now!  (e.g. \'Cathey, turn on the light.\')')
    audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE,
                   channels=1, dtype='float32', device=DEVICE)
    sd.wait()
    print('Done.\n')
    energy = float(np.mean(np.abs(audio)))
    print(f'Energy level : {energy:.6f}')
    if energy < 0.003:
        print('⚠  Very quiet — check Windows Settings → System → Sound → Input.')
    else:
        print('✓  Audio captured.')
    audio_arr = audio

else:
    try:
        import pyaudio, numpy as np
        print('Audio backend : pyaudio (fallback) ✓')
    except ImportError:
        print('[ERROR] Neither sounddevice nor pyaudio is installed.')
        print('Install: pip install sounddevice   OR   pip install pyaudio')
        raise
    CHUNK = 1024
    pa = pyaudio.PyAudio()
    print('\n── Available microphone devices (pyaudio) ───────────────────')
    for i in range(pa.get_device_count()):
        info = pa.get_device_info_by_index(i)
        if info['maxInputChannels'] > 0:
            print(f"  [{i:2d}] {info['name']}")
    print()
    DEVICE_INDEX = None
    stream = pa.open(format=pyaudio.paInt16, channels=1, rate=SAMPLE_RATE,
                     input=True, frames_per_buffer=CHUNK, input_device_index=DEVICE_INDEX)
    print(f'Recording {DURATION}s via pyaudio — speak now!')
    frames = [stream.read(CHUNK) for _ in range(int(SAMPLE_RATE / CHUNK * DURATION))]
    stream.stop_stream(); stream.close(); pa.terminate()
    print('Done.\n')
    raw = b''.join(frames)
    audio_arr = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    audio_arr = audio_arr.reshape(-1, 1)
    energy = float(np.mean(np.abs(audio_arr)))
    print(f'Energy level : {energy:.6f}')
    if energy < 0.003:
        print('⚠  Very quiet — check Windows Settings → Sound → Recording.')
    else:
        print('✓  Audio captured.')

# ── Common: STT + Agent ───────────────────────────────────────────────────────
try:
    from audio import STTModel
    stt = STTModel()
    text = stt.transcribe(audio_arr)
    print(f'\nTranscription: {text!r}')
    if text:
        print('\n── Agent result ─────────────────────────────────────────────')
        result = agent.handle(text, verbose=True)
        print(f"valid={result['valid']}  reason={result['reason']}  exec={result.get('execution')}")
except Exception as e:
    print(f'\n[STT/Agent ERROR] {e}')
